In [8]:
# Section 1: Import Libraries & Load Data
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest

df = pd.read_csv("marketing_AB.csv")
df.columns = df.columns.str.replace(' ', '_').str.lower()
print(df.shape)
print(df.head())

(588101, 7)
   unnamed:_0  user_id test_group  converted  total_ads most_ads_day  \
0           0  1069124         ad      False        130       Monday   
1           1  1119715         ad      False         93      Tuesday   
2           2  1144181         ad      False         21      Tuesday   
3           3  1435133         ad      False        355      Tuesday   
4           4  1015700         ad      False        276       Friday   

   most_ads_hour  
0             20  
1             22  
2             18  
3             10  
4             14  


In [9]:
# Section 2: EDA

# 2-1. Group distribution
print("Group Distribution:")
print(df['test_group'].value_counts())
print()

# 2-2. Overall conversion rate by group
print("Conversion Rate by Group:")
print(df.groupby('test_group')['converted'].agg(['sum', 'count', 'mean']).rename(columns={'sum': 'conversions', 'count': 'total_users', 'mean': 'conversion_rate'}))
print()

# 2-3. Check missing values
print("Missing Values:")
print(df.isnull().sum())

Group Distribution:
test_group
ad     564577
psa     23524
Name: count, dtype: int64

Conversion Rate by Group:
            conversions  total_users  conversion_rate
test_group                                           
ad                14423       564577         0.025547
psa                 420        23524         0.017854

Missing Values:
unnamed:_0       0
user_id          0
test_group       0
converted        0
total_ads        0
most_ads_day     0
most_ads_hour    0
dtype: int64


In [10]:
# 2-4. Conversion rate by day of week
day_analysis = df.groupby(['test_group', 'most_ads_day']).agg(
    total_users=('converted', 'count'),
    conversions=('converted', 'sum')
).reset_index()

day_analysis['conversion_rate'] = round(day_analysis['conversions'] / day_analysis['total_users'] * 100, 2)

# Sort by day order
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_analysis['most_ads_day'] = pd.Categorical(day_analysis['most_ads_day'], categories=day_order, ordered=True)
day_analysis = day_analysis.sort_values(['test_group', 'most_ads_day'])

print(day_analysis)

   test_group most_ads_day  total_users  conversions  conversion_rate
1          ad       Monday        83571         2778             3.32
5          ad      Tuesday        74572         2270             3.04
6          ad    Wednesday        77418         1963             2.54
4          ad     Thursday        79077         1711             2.16
0          ad       Friday        88805         1995             2.25
2          ad     Saturday        78802         1679             2.13
3          ad       Sunday        82332         2027             2.46
8         psa       Monday         3502           79             2.26
12        psa      Tuesday         2907           42             1.44
13        psa    Wednesday         3490           55             1.58
11        psa     Thursday         3905           79             2.02
7         psa       Friday         3803           62             1.63
9         psa     Saturday         2858           40             1.40
10        psa       

In [11]:
# 2-5. Conversion rate by hour
hour_analysis = df.groupby(['test_group', 'most_ads_hour']).agg(
    total_users=('converted', 'count'),
    conversions=('converted', 'sum')
).reset_index()

hour_analysis['conversion_rate'] = round(hour_analysis['conversions'] / hour_analysis['total_users'] * 100, 2)
hour_analysis = hour_analysis.sort_values(['test_group', 'most_ads_hour'])

print(hour_analysis)

   test_group  most_ads_hour  total_users  conversions  conversion_rate
0          ad              0         5309          102             1.92
1          ad              1         4615           62             1.34
2          ad              2         5152           39             0.76
3          ad              3         2590           27             1.04
4          ad              4          694           11             1.59
5          ad              5          742           16             2.16
6          ad              6         1985           46             2.32
7          ad              7         6168          114             1.85
8          ad              8        16968          337             1.99
9          ad              9        29802          582             1.95
10         ad             10        37454          818             2.18
11         ad             11        44149          992             2.25
12         ad             12        45238         1092          

In [12]:
# 2-6. Total ads exposure vs conversion rate
df['ads_bucket'] = pd.cut(df['total_ads'], 
                           bins=[0, 10, 50, 100, 500, 2000],
                           labels=['1-10', '11-50', '51-100', '101-500', '500+'])

ads_analysis = df.groupby(['test_group', 'ads_bucket']).agg(
    total_users=('converted', 'count'),
    conversions=('converted', 'sum')
).reset_index()

ads_analysis['conversion_rate'] = round(ads_analysis['conversions'] / ads_analysis['total_users'] * 100, 2)
print(ads_analysis)

  test_group ads_bucket  total_users  conversions  conversion_rate
0         ad       1-10       249499          813             0.33
1         ad      11-50       248875         4696             1.89
2         ad     51-100        44149         5135            11.63
3         ad    101-500        21488         3680            17.13
4         ad       500+          565           99            17.52
5        psa       1-10        11276           45             0.40
6        psa      11-50         9385          148             1.58
7        psa     51-100         1853          107             5.77
8        psa    101-500          991          118            11.91
9        psa       500+           19            2            10.53


/var/folders/qt/dj7wj87x7nl411c7m12g8qjr0000gn/T/ipykernel_51767/2072102143.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  ads_analysis = df.groupby(['test_group', 'ads_bucket']).agg(


In [13]:
# Section 3: A/B Test

# Hypothesis
# H0: There is no significant difference in conversion rates between ad and psa groups
# H1: Ad group has a significantly higher conversion rate than psa group

# Prepare data
n_ad = len(df[df['test_group'] == 'ad'])
n_psa = len(df[df['test_group'] == 'psa'])
conv_ad = df[df['test_group'] == 'ad']['converted'].sum()
conv_psa = df[df['test_group'] == 'psa']['converted'].sum()

print(f"Ad group: {conv_ad} conversions out of {n_ad} ({conv_ad/n_ad*100:.2f}%)")
print(f"PSA group: {conv_psa} conversions out of {n_psa} ({conv_psa/n_psa*100:.2f}%)")

# Z-test
from statsmodels.stats.proportion import proportions_ztest

z_stat, p_value = proportions_ztest([conv_ad, conv_psa], [n_ad, n_psa], alternative='larger')

print(f"\nZ-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print()
print(f"Conclusion: {'Reject H0 - Ad significantly outperforms PSA' if p_value < 0.05 else 'Fail to reject H0 - No significant difference'}")

Ad group: 14423 conversions out of 564577 (2.55%)
PSA group: 420 conversions out of 23524 (1.79%)

Z-statistic: 7.3701
P-value: 0.0000

Conclusion: Reject H0 - Ad significantly outperforms PSA
